# M04-03 — Segmentación

Referencia de validación. El alumno trabaja en `notebooks/alumno/M04-03-segmentacion.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, sum as fsum, countDistinct, when, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile
spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = fact.join(customers, "customer_id", "inner").where(col("is_billable"))
customer_gmv = sales.groupBy("customer_id", "country", "segment").agg(
    fsum("gmv_line").alias("gmv"), countDistinct("order_id").alias("orders")
)
print(customer_gmv.count())
assert customer_gmv.count() == 211
banded = customer_gmv.withColumn(
    "value_band",
    when(col("gmv") < 1000, lit("low")).when(col("gmv") < 3000, lit("mid")).otherwise(lit("high")),
)
banded.groupBy("value_band").count().orderBy("value_band").show()
assert banded.count() == 211
banded.write.mode("overwrite").parquet(str(STAGING / "customer_gmv"))
w = Window.orderBy(col("gmv"))
customer_gmv.withColumn("q", ntile(5).over(w)).groupBy("q").count().orderBy("q").show()
print("M04-03 OK")
